<a href="https://colab.research.google.com/github/rah-ds/Cloud-Autoscaling-using-RL/blob/bmcgregor%2Fsimulator-refactor/notebooks/Experiment_Policy_ActorCritic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df_usage = pd.read_csv('/content/drive/MyDrive/df_usage.csv')
display(df_usage.head())

,time_window,avg_cpu,avg_mem,active_machines
0,1970-01-01 00:05:00+00:00,0.006623,0.004912,9525
1,1970-01-01 00:06:00+00:00,0.003254,0.002733,3805
2,1970-01-01 00:07:00+00:00,0.003070,0.002770,4167
3,1970-01-01 00:08:00+00:00,0.001950,0.001823,4338
4,1970-01-01 00:09:00+00:00,0.001689,0.001468,5545


In [ ]:
import requests

raw_url = "https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/bmcgregor/simulator-refactor/scripts/autoscaling_env.py"
local_filename = "autoscaling_env.py"

r = requests.get(raw_url)
r.raise_for_status()  # ensures you get an error if download fails

with open(local_filename, "wb") as f:
    f.write(r.content)

print("Downloaded:", local_filename)


Downloaded: autoscaling_env.py


In [ ]:
#df_usage_train = df_usage.head(1000)
print(f"df_usage now has {df_usage.shape[0]} rows and {df_usage.shape[1]} columns.")

df_usage now has 10000 rows and 4 columns.


In [ ]:
import importlib
import autoscaling_env # Ensure the module is loaded if not already

# Reload the module to pick up the changes
importlib.reload(autoscaling_env)

# Re-import the class and re-initialize the environment
from autoscaling_env import AutoScalingEnv
env = AutoScalingEnv(df_usage_train)

print('Resetting the environment...')
initial_state, initial_info = env.reset()
print(f'Initial State: {initial_state}')
print(f'Initial Info: {initial_info}')

import random
action = random.randint(0, env.action_space.n - 1)
print(f'Taking random action: {action}')

next_state, reward, terminated, truncated, info = env.step(action)

print(f'\nAfter one step:')
print(f'Next State: {next_state}')
print(f'Reward: {reward}')
print(f'Terminated: {terminated}')
print(f'Truncated: {truncated}')
print(f'Info: {info}')

Resetting the environment...
Initial State: [6.6225836e-03 4.9124165e-03 1.0000000e+01]
Initial Info: {'initial_capacity': 10}
Taking random action: 2

After one step:
Next State: [3.2542923e-03 2.7331815e-03 1.1000000e+01]
Reward: -54.50210704835946
Terminated: False
Truncated: False
Info: {'current_capacity': 11, 'utilization': np.float64(5.734555186214497), 'estimated_total_cpu_load': np.float64(63.080107048359466), 'reward_components': {'cost_penalty': -0.022, 'sla_penalty': np.float64(-49.345551862144966), 'util_deviation_penalty': np.float64(-5.134555186214497)}}


#Actor Critic

 # 1. Define the Actor-Critic Network
We'll create a single neural network with two heads: one for the policy (actor) and one for the value function (critic). This allows for shared layers and potentially more efficient learning.

In [ ]:
import torch.nn.functional as F

# Define the Actor-Critic Network
class ActorCriticNetwork(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super(ActorCriticNetwork, self).__init__()

        self.shared_layers = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU()
        )

        self.actor_head = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1) # Softmax for probabilities over the last dimension
        )

        self.critic_head = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1) # Output a single value for the state-value function
        )

    def forward(self, x):
        shared_features = self.shared_layers(x)
        action_probs = self.actor_head(shared_features)
        state_value = self.critic_head(shared_features)
        return action_probs, state_value

# Initialize the Actor-Critic model
actor_critic_model = ActorCriticNetwork(observation_space_dim, action_space_dim).to(device)
print(actor_critic_model)

NameError: name 'nn' is not defined

### 2. Implement the Actor-Critic Training Step
Now, let's redefine the `train_step` function to handle both actor and critic updates. We'll use the Temporal Difference (TD) error as the advantage for the actor, and train the critic using Mean Squared Error (MSE).

In [ ]:
actor_critic_optimizer = optim.Adam(actor_critic_model.parameters(), lr=0.001)

def train_step_actor_critic(states, actions, rewards, next_states, dones, gamma=0.99):
    actor_critic_optimizer.zero_grad()

    # Convert inputs to PyTorch tensors and move to device
    states_t = torch.tensor(states, dtype=torch.float32).to(device)
    actions_t = torch.tensor(actions, dtype=torch.int64).to(device)
    rewards_t = torch.tensor(rewards, dtype=torch.float32).to(device)
    next_states_t = torch.tensor(next_states, dtype=torch.float32).to(device)
    dones_t = torch.tensor(dones, dtype=torch.float32).to(device)

    # Get action probabilities and state values from the model
    action_probs, state_values = actor_critic_model(states_t)

    # Get next state values for TD target calculation
    with torch.no_grad():
        _, next_state_values = actor_critic_model(next_states_t)
        # If done, next state value is 0
        next_state_values = next_state_values * (1 - dones_t.unsqueeze(1))

    # Calculate TD target
    td_target = rewards_t.unsqueeze(1) + gamma * next_state_values

    # Calculate TD Error (advantage)
    td_error = td_target - state_values

    # Critic Loss (MSE between state_values and TD_target)
    critic_loss = F.mse_loss(state_values, td_target.detach())

    # Actor Loss (Policy Gradient with TD_error as advantage)
    m = torch.distributions.Categorical(action_probs)
    log_probs = m.log_prob(actions_t)
    actor_loss = -torch.sum(log_probs * td_error.detach())

    # Total loss
    total_loss = actor_loss + critic_loss

    # Perform backpropagation
    total_loss.backward()
    actor_critic_optimizer.step()

    return total_loss.item(), actor_loss.item(), critic_loss.item()

### 3. Training Loop for Actor-Critic Agent
Now, we'll implement the training loop for our Actor-Critic agent, which will utilize the `actor_critic_model` and `train_step_actor_critic` function.

In [ ]:
num_episodes_ac = 200 # Number of episodes for Actor-Critic training
gamma_ac = 0.99 # Discount factor

episode_rewards_ac = []
actor_losses = []
critic_losses = []

for episode in range(num_episodes_ac):
    states, actions, rewards, next_states, dones = [], [], [], [], []
    state, info = env.reset()
    done = False
    episode_reward = 0
    steps_in_episode = 0

    while not done:
        # Convert state to PyTorch tensor, add batch dimension, and move to device
        state_input = torch.tensor(state[np.newaxis, :], dtype=torch.float32).to(device)

        # Predict action probabilities from the actor-critic model
        action_probs, _ = actor_critic_model(state_input)

        # Sample an action from the distribution
        m = torch.distributions.Categorical(action_probs)
        action = m.sample().item()

        # Take action in the environment
        next_state, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

        states.append(state)
        actions.append(action)
        rewards.append(reward)
        next_states.append(next_state)
        dones.append(float(done))

        state = next_state
        episode_reward += reward
        steps_in_episode += 1

    # Normalize episode reward by steps
    normalized_episode_reward_ac = episode_reward / steps_in_episode if steps_in_episode > 0 else 0

    # Convert lists to NumPy arrays, then to PyTorch tensors for train_step_actor_critic
    states_np = np.array(states, dtype=np.float32)
    actions_np = np.array(actions, dtype=np.int32)
    rewards_np = np.array(rewards, dtype=np.float32)
    next_states_np = np.array(next_states, dtype=np.float32)
    dones_np = np.array(dones, dtype=np.float32)

    # Perform a training step for Actor-Critic
    total_loss, actor_loss, critic_loss = train_step_actor_critic(
        states_np, actions_np, rewards_np, next_states_np, dones_np, gamma=gamma_ac
    )

    episode_rewards_ac.append(normalized_episode_reward_ac)
    actor_losses.append(actor_loss)
    critic_losses.append(critic_loss)

    if episode % 10 == 0:
        print(f"Episode {episode}: Total Reward = {episode_reward:.2f}, Normalized Reward = {normalized_episode_reward_ac:.2f}, Actor Loss = {actor_loss:.4f}, Critic Loss = {critic_loss:.4f}, Avg Normalized Reward (last 10) = {np.mean(episode_rewards_ac[-10:]):.2f}")

print("\nActor-Critic Training finished!")

### 4. Evaluate the Trained Actor-Critic Agent
Let's visualize the training progress and run a final evaluation episode to see how the Actor-Critic agent performs.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(episode_rewards_ac)
plt.title('Actor-Critic Training: Episode Normalized Rewards Over Time')
plt.xlabel('Episode')
plt.ylabel('Normalized Total Reward (per step)')
plt.grid(True)
plt.show()

print("\n--- Evaluation of Trained Actor-Critic Agent ---")
state_ac_eval, info_ac_eval = env.reset()
done_ac_eval = False
episode_reward_ac_eval = 0
steps_in_ac_eval_episode = 0

while not done_ac_eval:
    state_input_ac_eval = torch.tensor(state_ac_eval[np.newaxis, :], dtype=torch.float32).to(device)

    # Get action probabilities from the actor-critic model
    with torch.no_grad(): # Disable gradient calculations for inference
        action_probs_ac_eval, _ = actor_critic_model(state_input_ac_eval)

    # Take the most probable action for evaluation
    action_ac_eval = torch.argmax(action_probs_ac_eval).item()

    next_state_ac_eval, reward_ac_eval, terminated_ac_eval, truncated_ac_eval, info_ac_eval = env.step(action_ac_eval)
    done_ac_eval = terminated_ac_eval or truncated_ac_eval

    state_ac_eval = next_state_ac_eval
    episode_reward_ac_eval += reward_ac_eval
    steps_in_ac_eval_episode += 1

normalized_ac_eval_reward = episode_reward_ac_eval / steps_in_ac_eval_episode if steps_in_ac_eval_episode > 0 else 0
print(f"Evaluation Episode (Actor-Critic): Total Reward = {episode_reward_ac_eval:.2f}, Normalized Reward (per step) = {normalized_ac_eval_reward:.2f}")